changes we made to train_controlnet.py

✅ 1. load_dataset → load_from_disk
Original:


dataset = load_dataset(
    args.train_data_dir,
    cache_dir=args.cache_dir,
)
Changed To:


from datasets import load_from_disk
dataset = load_from_disk(args.train_data_dir)
✅ 2. Removed dataset["train"]
Original usage (incorrect for load_from_disk):


column_names = dataset["train"].column_names
Changed To:

column_names = dataset.column_names

🔁 Why this matters:
load_from_disk() loads a dataset already split, so dataset is the training data itself.

There’s no need to access .train or any other key.

Trying dataset["train"] will throw a KeyError.

In [1]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
print("ControlNet import OK ✅")
print("ControlNet pipeline ✅")

In [3]:
!pip install wandb tqdm pandas opencv-python

In [2]:
import os

os.makedirs("/workspace/controlnet_trained_1.2", exist_ok=True)
os.makedirs("/workspace/samples", exist_ok=True)

In [3]:
# ✅ Accelerate ile eğitim ayarları

# Output directory (Model buraya kaydolacak)
output_dir = "/workspace/controlnet_trained_1.2"

validation_image_paths = [
    "/workspace/samples/sample1.png",
    "/workspace/samples/sample2.png",
]

validation_prompts = [
    "dense street layout adapted to mountainous terrain",  # istediğin layout türü
]

In [4]:
%cd /workspace/diffusers/examples/controlnet

/workspace/diffusers/examples/controlnet


In [9]:
# ✅ STEP 1: Clean conflicting installations (optional but recommended)
!pip uninstall -y diffusers huggingface_hub torch torchvision torchaudio xformers
!rm -rf /usr/local/lib/python3.10/dist-packages/diffusers*
!rm -rf /usr/local/lib/python3.10/dist-packages/huggingface_hub*
!rm -rf /usr/local/lib/python3.10/dist-packages/torch*
!rm -rf ~/.cache

In [10]:

# ✅ STEP 2: Install compatible PyTorch + CUDA
!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

# ✅ STEP 3: Install other packages
!pip install xformers==0.0.21
!pip install transformers==4.35.2 accelerate==0.24.1
!pip install datasets wandb

# ✅ STEP 4: Clone and install diffusers from source
!git clone https://github.com/huggingface/diffusers
%cd diffusers
!pip install -e .

In [11]:

# ✅ STEP 5: Patch PIL ExifTags bug (optional but critical for decoding images)
from PIL import Image, ExifTags
Image.ExifTags = ExifTags

In [12]:
# ✅ STEP 6: Verify everything
import torch, torchvision, torchaudio
import diffusers, transformers, accelerate
print("torch:", torch.__version__)
print("diffusers:", diffusers.__path__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

torch: 2.0.1+cu118
diffusers: ['/workspace/diffusers/src/diffusers']
transformers: 4.35.2
accelerate: 0.24.1


In [22]:
!mkdir -p /workspace/diffusers/data/train/overlap
!mkdir -p /workspace/diffusers/data/train/elevation
!mkdir -p /workspace/diffusers/data/train/  # already exists


In [ ]:
# ✅ 1. Hugging Face login (programmatically)
from huggingface_hub import login
hf_token = "HF_TOKEN"  # Your token
login(token=hf_token)
print("✅ Hugging Face login completed")

# ✅ 2. WandB login (programmatically)
import wandb
wandb_token = "WANDB_TOKEN"  # Your token
wandb.login(key=wandb_token)
print("✅ WandB login completed")

# ✅ 3. Clone dataset using Git LFS
import os

os.system("apt-get install git-lfs -y")
os.system("git lfs install")
os.system("git clone https://huggingface.co/datasets/SalvadorCB/NASADEM_DATASET")

print("✅ Dataset cloned via git-lfs")

# ✅ 4. Optional: show structure
os.system("ls -l NASADEM_DATASET")



✅ Hugging Face login completed
✅ WandB login completed
Reading package lists...
Building dependency tree...
Reading state information...
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 143 not upgraded.
Updated git hooks.
Git LFS initialized.


Cloning into 'NASADEM_DATASET'...


✅ Dataset cloned via git-lfs
total 2172404
-rw-r--r-- 1 root root        316 May  1 15:58 README.md
-rw-r--r-- 1 root root 2224535777 May  1 15:58 train.zip


0

In [33]:
!unzip -o NASADEM_DATASET/train.zip -d /workspace/fixed_dataset


In [40]:
import os
import pandas as pd
from datasets import Dataset, Features, Image, Value

# Load metadata CSV
csv_path = "/workspace/fixed_dataset/train/metadata.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()  # Remove any accidental spaces

# Define image paths using correct column names
base_path = "/workspace/fixed_dataset/train"
df["elevation"] = df["elevation_file_name"].apply(lambda x: os.path.join(base_path, x))
df["overlap"]   = df["overlap_file_name"].apply(lambda x: os.path.join(base_path, x))

df["overlap_description"] = df["overlap_description"].astype(str)

# Define Hugging Face dataset features
features = Features({
    "elevation": Image(),
    "overlap": Image(),
    "overlap_description": Value("string")
})

# Create Hugging Face dataset
dataset = Dataset.from_pandas(df[["elevation", "overlap", "overlap_description"]], features=features)

print(dataset[0])





{'elevation': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1024x1024 at 0x7F7B8E15DE70>, 'overlap': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=1024x1024 at 0x7F7B8E15E080>, 'overlap_description': 'The image depicts a varied urban street network overlaid on a grayscale topographic map, where the streets adapt dynamically to the underlying terrain. In the central portion of the map—characterized by lighter shades indicating elevated terrain—the street network exhibits a dispersed and winding structure, curving to follow natural ridges and contours. These hillside areas feature fewer intersections and irregular road geometry, a sign of adaptation to challenging topography. In contrast, lower-elevation zones, indicated by darker shades, particularly toward the southern and western sections, display a denser grid-like layout with frequent intersections, reflecting flat terrain more conducive to structured urban planning. Major arterial roads can be seen traversing both l

In [42]:
dataset.save_to_disk("/workspace/fixed_dataset_hf")


Saving the dataset (0/4 shards):   0%|          | 0/1365 [00:00<?, ? examples/s]

In [49]:
import os

val_path = "/workspace/samples/sample1.png.png"

if os.path.exists(val_path):
    print(f"✅ Validation image found at: {val_path}")
else:
    print(f"❌ Validation image NOT found at: {val_path}")


✅ Validation image found at: /workspace/samples/sample1.png.png


In [50]:
!accelerate launch examples/controlnet/train_controlnet.py \
  --pretrained_model_name_or_path=runwayml/stable-diffusion-v1-5 \
  --output_dir=/workspace/controlnet_trained \
  --train_data_dir=/workspace/fixed_dataset_hf \
  --conditioning_image_column=elevation \
  --image_column=overlap \
  --caption_column=overlap_description \
  --resolution=1024 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=8 \
  --learning_rate=1e-5 \
  --num_train_epochs=3 \
  --checkpointing_steps=250 \
  --validation_steps=250 \
  --mixed_precision=fp16 \
  --enable_xformers_memory_efficient_attention \
  --gradient_checkpointing \
  --report_to=wandb \
  --tracker_project_name=controlnet-topo-street \
  --validation_image=/workspace/samples/sample1.png.png \
  --validation_prompt="dense street layout adapted to mountainous terrain" \
  --logging_dir=/workspace/logs


In [52]:
from huggingface_hub import create_repo, upload_folder

repo_id = "MisraSerenayy/controlnet-topo-street-1.3"  # ✅ replace with your desired repo name

# Create repo (if it doesn't already exist)
create_repo(repo_id, exist_ok=True)

# Upload the model folder
upload_folder(
    repo_id=repo_id,
    folder_path=output_dir,
    path_in_repo="",
    commit_message="Upload fine-tuned ControlNet model"
)
print(f"🚀 Model uploaded to https://huggingface.co/{repo_id}")

No files have been modified since last commit. Skipping to prevent empty commit.


🚀 Model uploaded to https://huggingface.co/MisraSerenayy/controlnet-topo-street-1.3


In [53]:
!cat .gitattributes


cat: .gitattributes: No such file or directory


In [54]:
%cd /workspace/controlnet_trained


/workspace/controlnet_trained


In [55]:
!echo "*.safetensors filter=lfs diff=lfs merge=lfs -text" > .gitattributes


In [56]:
!echo "*.bin filter=lfs diff=lfs merge=lfs -text" >> .gitattributes


In [57]:
from huggingface_hub import upload_folder

upload_folder(
    repo_id="MisraSerenayy/controlnet-topo-street-1.3",
    folder_path="/workspace/controlnet_trained",
    path_in_repo="",
    commit_message="Fix: Add .gitattributes for proper Git LFS tracking"
)


random_states_0.pkl:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

optimizer.bin:   0%|          | 0.00/2.89G [00:00<?, ?B/s]

Upload 11 LFS files:   0%|          | 0/11 [00:00<?, ?it/s]

scaler.pt:   0%|          | 0.00/557 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

...

random_states_0.pkl:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

scaler.pt:   0%|          | 0.00/557 [00:00<?, ?B/s]

scheduler.bin:   0%|          | 0.00/563 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/MisraSerenayy/controlnet-topo-street-1.3/commit/6736fa6e854ff2348c5f243e547c0864ba888ceb', commit_message='Fix: Add .gitattributes for proper Git LFS tracking', commit_description='', oid='6736fa6e854ff2348c5f243e547c0864ba888ceb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/MisraSerenayy/controlnet-topo-street-1.3', endpoint='https://huggingface.co', repo_type='model', repo_id='MisraSerenayy/controlnet-topo-street-1.3'), pr_revision=None, pr_num=None)